In [0]:
# -----------------------------------------------------------------------------
# CELL 2: LOGGING SETUP
# Why: Using Python's logging module instead of print() lets you:
#   - Control log level (DEBUG vs INFO vs WARNING)
#   - Add timestamps automatically
#   - Redirect logs to Azure Monitor or a file later with zero code change
# -----------------------------------------------------------------------------
 
logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt= "%Y-%m-%dT%H:%M:%SZ"
)
logger = logging.getLogger("landing_ingestion")

In [0]:
import requests   # HTTP calls to CoinGecko REST API
import json       # Serialize Python dicts → JSON strings before saving
import uuid       # Generate a unique ID (UUID4) for each pipeline run
import time       # Sleep between API calls to respect rate limits
import logging    # Structured logging (better than bare print for production)
 
from datetime import datetime, timezone   # Always work in UTC — never local time
from typing   import Any, Dict, List, Optional
# -----------------------------------------------------------------------------
# CELL 3: CONFIGURATION — Read from Key Vault via Databricks Secret Scope
#
# WHY KEY VAULT?
#   Never hardcode API keys or storage credentials in a notebook. If the
#   notebook is shared, exported, or accidentally committed to Git, those
#   secrets would be exposed. Key Vault + Secret Scope means the values are
#   never visible in the notebook — not even to you when you open it.
#
# HOW DBUTILS.SECRETS WORKS:
#   dbutils.secrets.get(scope, key) makes a secure call to the Databricks
#   Secret Store backend (which is linked to your Azure Key Vault). The
#   returned string is redacted in any logs or outputs — it shows as [REDACTED].
# -----------------------------------------------------------------------------
 
# ── Secrets (never printed, never logged) ────────────────────────────────────
API_KEY   = dbutils.secrets.get(scope="keyvaulthp11", key="coingecko-api")
ADLS_NAME = "adlsnewhp"
 
# ── API Configuration ─────────────────────────────────────────────────────────
BASE_URL          = "https://api.coingecko.com/api/v3"
 
# How many coins to fetch OHLC + market_chart for.
# !! IMPORTANT FOR DEMO PLAN USERS !!
# CoinGecko Demo plan allows ~30 API calls/minute.
# With TOP_N_COINS = 50, we make 50 × 2 = 100 calls for OHLC + market_chart.
# The sleep between calls (see RATE_LIMIT_SLEEP) makes the total ~2 minutes.
# Start with 10 for your first test run, then increase to 50 for production.
TOP_N_COINS       = 50        # Change to 10 for initial testing
 
# Seconds to sleep between per-coin API calls to respect rate limits.
# At 1.2s per coin × 50 coins × 2 endpoints = ~2 min total for these loops.
RATE_LIMIT_SLEEP  = 1.2
 
# Max retries if an API call fails (e.g., temporary 429 Too Many Requests)
MAX_RETRIES       = 3
 
# Seconds to wait before retrying after a failed call
RETRY_BACKOFF_SEC = 5
 
# ── ADLS Path Construction ────────────────────────────────────────────────────
# abfss:// = Azure Blob File System Secure protocol (required for ADLS Gen2)
# "landing" = container name in your storage account
ADLS_ROOT = f"abfss://capstone@{ADLS_NAME}.dfs.core.windows.net"
 
# ── Run-level Metadata ────────────────────────────────────────────────────────
# These two values are generated ONCE per notebook run and stamped on every
# file saved. This is critical for data lineage — you can trace any Bronze
# Delta row back to the exact Landing file and run that produced it.
RUN_ID   = str(uuid.uuid4())                    # e.g., "3f2a1b9c-..."
NOW_UTC  = datetime.now(timezone.utc)           # Capture once; use everywhere
 
# Pre-compute path fragments used in every file path
DATE_PATH = NOW_UTC.strftime("%Y/%m/%d")        # e.g., "2024/11/15"
TS_SUFFIX = NOW_UTC.strftime("%Y%m%d_%H%M%S")  # e.g., "20241115_060000"
 
logger.info("=" * 70)
logger.info("Landing Ingestion Started")
logger.info(f"  Pipeline Run ID : {RUN_ID}")
logger.info(f"  Run Timestamp   : {NOW_UTC.isoformat()}")
logger.info(f"  ADLS Root       : {ADLS_ROOT}")
logger.info(f"  Top N Coins     : {TOP_N_COINS}")
logger.info("=" * 70)
 